# Анализа трошкова за захтев

Овај нотебоок демонстрира како направити агенте који користе додатке за обраду трошкова путовања са локалних слика рачуна, генеришу имејл за захтев трошкова, и визуализују податке о трошковима користећи питачни дијаграм. Агенти динамички бирају функције у зависности од контекста задатка.

Кораци:
1. OCR агент обрађује локалну слику рачуна и извлачи податке о трошковима путовања.
2. Email агент генерише имејл за захтев трошкова.

### Пример сценарија за трошкове путовања:
Замислите да сте запослени који путује на пословни састанак у други град. Ваша компанија има политику да надокнади све разумне трошкове везане за путовање. Ево прегледа потенцијалних трошкова путовања:
- Транспорт:
Цена авионске карте за повратно путовање из вашег родног града до града одредишта.
Такси или услуге вожње до и од аеродрома.
Локални превоз у граду одредишта (као што су јавни превоз, изнајмљивање аутомобила или такси).

- Смештај:
Боравак у хотелу три ноћи у бизнис хотелу средњег ранга близу места састанка.

- Оброци:
Дневни износ за оброке за доручак, ручак и вечеру, у складу са политиком дневница компаније.

- Разни трошкови:
Паркинг на аеродрому.
Трошкови приступа интернету у хотелу.
Напојнице или мање услужне накнаде.

- Документација:
Предајете све рачуне (летови, такси, хотел, оброци итд.) и попуњени извештај о трошковима ради надокнаде.


## Увези потребне библиотеке

Увези неопходне библиотеке и модуле за свеску.


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import base64
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## Definišite Modele Troškova

 Kreirajte Pydantic model za pojedinačne troškove i klasu ExpenseFormatter za konvertovanje korisničkog upita u strukturirane podatke o troškovima.

 Svaki trošak će biti predstavljen u formatu:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Дефинисање алата - Генерисање имејла

Креирајте функцију алата за генерисање имејла за подношење захтева за накнаду трошкова.  
- Овај алат користи `@tool` декоратор из Microsoft Agent Framework-а.  
- Израчунава укупну суму трошкова и форматира детаље у тело имејла.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Алат за издвајање трошкова путовања са слика рачуна

Креирајте функцију алата за издвајање трошкова путовања са слика рачуна.
- Овај алат користи `@tool` декоратор из Microsoft Agent Framework-а.
- Он чита слику рачуна, кодира је у base64, и враћа data URI за агентову анализу.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Обрада трошкова

Дефинишите агенте и повежите их у секвенцијални ток рада користећи `WorkflowBuilder`.
- OCR агент извлачи структуриране податке о трошковима из слике рачуна користећи алат `load_receipt_image`.
- Емаил агент преузима издвојене податке и генерише професионалну емаил пријаву трошкова користећи алат `generate_expense_email`.
- `WorkflowBuilder` са `add_edge` креира секвенцијалну цевовод: OCR агент → Емаил агент.


In [ ]:
ocr_agent = client.as_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Главна функција

Направите секвенцијални ток рада и покрените га да бисте обрадили слику рачуна и генерисали имејл за захтев трошкова.


> **Напомена:** Овај ток рада тренутно прослеђује слику рачуна као base64 текст, што већина модела за разговор (укључујући gpt-4o) неће третирати као слику.  
> Такође, може прећи прозор контекста модела. Боље је покренути OCR са Azure AI Vision (или другим OCR алатом) и проследити само извучени текст, или преформатирати да се слика пошаље као порука `image_url`.  
> Ако желите само да избегнете грешке у конексту, покушајте са мањом сликом рачуна или моделом са већим прозором контекста.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:
Овај документ је преведен коришћењем услуге за аутоматски превод [Co-op Translator](https://github.com/Azure/co-op-translator). Иако тежимо тачности, имајте у виду да аутоматски преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитативним извором. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било каква неспоразума или погрешна тумачења која произилазе из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
